# Examine sv-evidence-extraction output

Basic loading and sanity-check tools for the PE/SR/RD tables produced by
a `query`- or `build-tables`-mode run: discover what result sets are on
disk, load one into a dict of DataFrames, label each row with the
sample's family role (child/father/mother, via the pedigree), and get a
quick per-class, per-sample row-count summary before doing any real
analysis.

`local_evidence_dir` in `local_config.json` points at wherever you've
downloaded results to locally (gitignored, like the other workspace
paths -- see `local_config.example.json`).


In [ ]:
# ==========================================
# IMPORTS
# ==========================================
import json
from pathlib import Path

import pandas as pd


In [ ]:
# ==========================================
# LOCAL CONFIG
# ==========================================
# Assumes the notebook is run from its default location (notebooks/);
# adjust REPO_ROOT if you've moved it.
REPO_ROOT = Path.cwd().parent
LOCAL_CONFIG_PATH = REPO_ROOT / "local_config.json"

with open(LOCAL_CONFIG_PATH) as fh:
    local_config = json.load(fh)

DATA_DIR = Path(local_config["local_evidence_dir"])
DATA_DIR


In [ ]:
# ==========================================
# DISCOVER AVAILABLE RESULT SETS
# ==========================================
def list_available_prefixes(data_dir, evidence_class="pe", fmt="parquet"):
    """List the distinct output_prefix values with evidence tables present in a directory.

    Parameters
    ----------
    data_dir : str or pathlib.Path
        Directory containing sv-evidence-extraction output files, named
        "<prefix>.{pe,sr,rd}.{tsv,parquet}".
    evidence_class : {"pe", "sr", "rd"}, default "pe"
        Which evidence class's files to key off of -- any one class is
        enough to discover the prefix, since all three (when present)
        share the same prefix.
    fmt : {"tsv", "parquet"}, default "parquet"
        Which file extension to look for.

    Returns
    -------
    list of str
        Sorted, unique output_prefix values found.
    """
    data_dir = Path(data_dir)
    suffix = f".{evidence_class}.{fmt}"
    return sorted({p.name[: -len(suffix)] for p in data_dir.glob(f"*{suffix}")})


In [ ]:
# ==========================================
# LOAD ONE RESULT SET
# ==========================================
EVIDENCE_CLASSES = ("pe", "sr", "rd")


def load_evidence_set(prefix, data_dir, fmt="parquet"):
    """Load the PE/SR/RD tables for one output_prefix into a dict of DataFrames.

    Parameters
    ----------
    prefix : str
        The output_prefix used when the evidence was extracted (matches
        the region_name from build_query_inputs.ipynb, e.g.
        "chr20_38104076_38108227_<child_id>").
    data_dir : str or pathlib.Path
        Directory containing the "<prefix>.{pe,sr,rd}.{tsv,parquet}" files.
    fmt : {"tsv", "parquet"}, default "parquet"
        Which file format to load -- parquet is smaller/faster locally;
        tsv is what the cluster-side WDL task also produces.

    Returns
    -------
    dict of str -> pandas.DataFrame
        Keys "pe", "sr", "rd". A missing file for a given class returns
        an empty DataFrame for that key rather than raising, so a
        partial result set still loads.
    """
    data_dir = Path(data_dir)
    reader = pd.read_parquet if fmt == "parquet" else (lambda p: pd.read_csv(p, sep="\t"))

    evidence = {}
    for evidence_class in EVIDENCE_CLASSES:
        path = data_dir / f"{prefix}.{evidence_class}.{fmt}"
        evidence[evidence_class] = reader(path) if path.exists() else pd.DataFrame()
    return evidence


In [ ]:
# ==========================================
# SANITY-CHECK SUMMARY
# ==========================================
def summarize_evidence(evidence, prefix=None):
    """Print a quick per-class, per-sample row-count sanity check.

    Worth running before any real analysis: an empty table can mean
    "genuinely no evidence" but can just as easily mean the extraction
    silently failed to access the source file (this happened once during
    development -- see OPERATIONS.md). RD in particular should
    essentially never be empty for a real sample/region, since bincov
    matrices are dense, so an empty RD table is a stronger red flag than
    an empty PE/SR table.

    Parameters
    ----------
    evidence : dict of str -> pandas.DataFrame
        As returned by `load_evidence_set`, optionally already passed
        through `label_evidence_relationships` -- if a "relationship"
        column is present, counts are broken out by it.
    prefix : str, optional
        Label to print above the summary, for readability when checking
        several result sets in a row.

    Returns
    -------
    pandas.DataFrame
        One row per (evidence_class, sample_id[, relationship]), with a
        "rows" count column -- meant to be read at a glance, not joined
        onto anything.
    """
    if prefix:
        print(f"=== {prefix} ===")

    rows = []
    for evidence_class, df in evidence.items():
        if df.empty:
            print(f"  {evidence_class.upper():>3}: EMPTY (0 rows) -- verify this is really \"no evidence\", not an access failure")
            continue

        group_cols = ["relationship", "sample_id"] if "relationship" in df.columns else ["sample_id"]
        counts = df.groupby(group_cols).size()
        print(f"  {evidence_class.upper():>3}: {len(df)} rows across {df['sample_id'].nunique()} sample(s)")
        for key, count in counts.items():
            row = dict(zip(group_cols, key if isinstance(key, tuple) else (key,)))
            row["evidence_class"] = evidence_class
            row["rows"] = count
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
# ==========================================
# PEDIGREE LOOKUP
# ==========================================
# Shared with build_query_inputs.ipynb via pedigree_utils.py.
from pedigree_utils import load_pedigree, label_evidence_relationships

df_ped = load_pedigree(local_config["ped_file_uri"])
df_ped.head()


## Example: load and summarize one result set


In [ ]:
# ==========================================
# EXAMPLE: LOAD, LABEL, AND SUMMARIZE ONE RESULT SET
# ==========================================
available = list_available_prefixes(DATA_DIR)
print(f"{len(available)} result set(s) found in {DATA_DIR}:")
for p in available:
    print(f"  {p}")

prefix = available[-1]
evidence = load_evidence_set(prefix, DATA_DIR)
evidence = label_evidence_relationships(evidence, df_ped)
summary = summarize_evidence(evidence, prefix=prefix)
summary


In [ ]:
evidence['pe']